In [ ]:
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torch.nn as nn

PROJECT_ROOT = Path.home() / 'sr_project'
CKPT         = PROJECT_ROOT / 'best_espcn_x4.pth'
TEST_DIR     = PROJECT_ROOT / 'Test'
SCALE        = 4

#  Model 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ESPCN(nn.Module):
    def __init__(self, scale=4):
        super().__init__()
        self.scale = scale
        self.net = nn.Sequential(
            nn.Conv2d(3,   64, 5, padding=2), nn.ReLU(True),
            nn.Conv2d(64, 64,  3, padding=1), nn.ReLU(True),
            nn.Conv2d(64, 32,  3, padding=1), nn.ReLU(True),
            nn.Conv2d(32, 3 * (scale ** 2), 3, padding=1)
        )
        self.pixel_shuffle = nn.PixelShuffle(scale)

    def forward(self, x):
        return torch.clamp(self.pixel_shuffle(self.net(x)), 0, 1)

def psnr(sr, hr):
    mse = F.mse_loss(sr.clamp(0, 1), hr.clamp(0, 1))
    return 10 * torch.log10(1.0 / (mse + 1e-8))

# Load model
model = ESPCN(scale=SCALE).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
model.eval()
print(f"✅ Model loaded  →  {device}\n")

#  Collect ALL images in the Test directory
lr_imgs = sorted([
    p for p in TEST_DIR.iterdir()
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
])

print(f"Found {len(lr_imgs)} image(s) in {TEST_DIR}\n")
assert len(lr_imgs) > 0, f"❌ No images found in {TEST_DIR}."

# Process each LR image 
for lr_path in lr_imgs:
    # Derive the original stem  (e.g. "photo_LR" → "photo")
    orig_stem = lr_path.stem[:-3]   # strip trailing "_LR"

    print(f"{'='*60}")
    print(f"Processing : {lr_path.name}")

    # Load LR image
    lr       = Image.open(lr_path).convert('RGB')
    lr_w, lr_h = lr.size
    out_w, out_h = lr_w * SCALE, lr_h * SCALE
    print(f"  LR size       : {lr_w}×{lr_h}")
    print(f"  Output size   : {out_w}×{out_h}  (×{SCALE})")

    #  Bicubic upscale (baseline) 
    bic = lr.resize((out_w, out_h), Image.BICUBIC)

    #  ESPCN upscale 
    lr_t = TF.to_tensor(lr).unsqueeze(0).to(device)
    with torch.no_grad():
        sr_t = model(lr_t).clamp(0, 1)
    sr = TF.to_pil_image(sr_t.squeeze(0).cpu())

    #  Save ESPCN SR image 
    sr_save_path = TEST_DIR / f"{orig_stem}_ESPCN{lr_path.suffix}"
    sr.save(sr_save_path)
    print(f"  ESPCN output  : {sr.size[0]}×{sr.size[1]}  → saved as {sr_save_path.name}")

    #  PSNR — only if original HR image exists alongside the LR 
    orig_path = TEST_DIR / f"{orig_stem}{lr_path.suffix}"
    psnr_line = ""
    if orig_path.exists():
        orig_img  = Image.open(orig_path).convert('RGB')
        hr_w      = (orig_img.size[0] // 4) * 4
        hr_h      = (orig_img.size[1] // 4) * 4
        hr_img    = orig_img.resize((hr_w, hr_h), Image.LANCZOS)
        hr_t      = TF.to_tensor(hr_img).unsqueeze(0).to(device)
        bic_t     = TF.to_tensor(bic).unsqueeze(0).to(device)
        p_bic     = psnr(bic_t, hr_t).item()
        p_sr      = psnr(sr_t,  hr_t).item()
        gain      = p_sr - p_bic
        

    #  3-panel comparison: LR Input | Bicubic ×4 | ESPCN ×4 
    # Show LR upscaled with NEAREST so pixels are clearly visible
    lr_display = lr.resize((out_w, out_h), Image.NEAREST)

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    fig.patch.set_facecolor('#0f0f0f')

    panels = [
        (lr_display, f'LR Input\n{lr_w}×{lr_h}',            '#aaaaaa'),
        (bic,        f'Bicubic ×{SCALE}\n{out_w}×{out_h}',   '#f5a623'),
        (sr,         f'ESPCN ×{SCALE}\n{out_w}×{out_h}',     '#4caf50'),
    ]

    for ax, (img, title, color) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(title, fontsize=13, color=color,
                     fontweight='bold', pad=10, linespacing=1.6)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(color)
            spine.set_linewidth(2.5)

    plt.suptitle(
        f'{orig_stem}  —  Super Resolution Comparison',
        fontsize=14, color='white', fontweight='bold', y=1.02
    )
    plt.tight_layout()

    compare_path = TEST_DIR / f"{orig_stem}_COMPARE{lr_path.suffix}"
    fig.savefig(compare_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    plt.close(fig)
    print(f"  Comparison    : saved as {compare_path.name}")

print(f"\n{'='*60}")
print(f"✅ All results saved in {TEST_DIR}")
print(f"{'='*60}")